# Notebook 06: Full Transformer Model

We assemble all components from previous notebooks into a complete **decoder-only transformer** (GPT-style):

```
Token Indices
    ↓
Character Embedding + Positional Encoding
    ↓
3x Transformer Block:
    ├── LayerNorm → Multi-Head Attention → Residual
    └── LayerNorm → FFN → Residual
    ↓
LayerNorm
    ↓
Linear Output Projection (→ vocab_size logits)
```

In [1]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

## 1. All Components (from previous notebooks)

In [2]:
def softmax(x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)


def cross_entropy_loss(logits, targets):
    """Cross-entropy loss.
    
    Args:
        logits: (N, C) raw scores
        targets: (N,) integer class labels
    Returns:
        loss: scalar
        dlogits: (N, C) gradient
    """
    N = logits.shape[0]
    probs = softmax(logits)
    log_probs = -np.log(probs[np.arange(N), targets] + 1e-9)
    loss = np.mean(log_probs)
    dlogits = probs.copy()
    dlogits[np.arange(N), targets] -= 1
    dlogits /= N
    return loss, dlogits


def causal_mask(seq_len):
    mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
    return mask[np.newaxis, np.newaxis, :, :]


def get_positional_encoding(max_seq_len, d_model):
    pe = np.zeros((max_seq_len, d_model))
    position = np.arange(max_seq_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe

In [3]:
class Embedding:
    def __init__(self, vocab_size, d_model):
        self.W = np.random.randn(vocab_size, d_model) * 0.02
        self.dW = None
        self.indices = None
    
    def forward(self, indices):
        self.indices = indices
        return self.W[indices]
    
    def backward(self, dout):
        self.dW = np.zeros_like(self.W)
        np.add.at(self.dW, self.indices, dout)


class LayerNorm:
    def __init__(self, d_model, eps=1e-5):
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)
        self.eps = eps
        self.dgamma = None
        self.dbeta = None
        self.x_hat = None
        self.std_inv = None
    
    def forward(self, x):
        mean = np.mean(x, axis=-1, keepdims=True)
        var = np.var(x, axis=-1, keepdims=True)
        self.std_inv = 1.0 / np.sqrt(var + self.eps)
        self.x_hat = (x - mean) * self.std_inv
        return self.gamma * self.x_hat + self.beta
    
    def backward(self, dout):
        D = dout.shape[-1]
        dout_flat = dout.reshape(-1, D)
        x_hat_flat = self.x_hat.reshape(-1, D)
        self.dgamma = np.sum(dout_flat * x_hat_flat, axis=0)
        self.dbeta = np.sum(dout_flat, axis=0)
        dx_hat = dout * self.gamma
        dx = self.std_inv * (
            dx_hat
            - np.mean(dx_hat, axis=-1, keepdims=True)
            - self.x_hat * np.mean(dx_hat * self.x_hat, axis=-1, keepdims=True)
        )
        return dx


class MultiHeadAttention:
    def __init__(self, d_model, n_heads):
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        scale = np.sqrt(2.0 / (d_model + self.d_k))
        self.W_Q = np.random.randn(d_model, d_model) * scale
        self.W_K = np.random.randn(d_model, d_model) * scale
        self.W_V = np.random.randn(d_model, d_model) * scale
        self.W_O = np.random.randn(d_model, d_model) * scale
        self.dW_Q = self.dW_K = self.dW_V = self.dW_O = None
        self.x = self.Q = self.K = self.V = None
        self.attn_weights = self.attn_output = None
    
    def _split_heads(self, x):
        B, T, D = x.shape
        return x.reshape(B, T, self.n_heads, self.d_k).transpose(0, 2, 1, 3)
    
    def _merge_heads(self, x):
        B, H, T, d_k = x.shape
        return x.transpose(0, 2, 1, 3).reshape(B, T, self.d_model)
    
    def forward(self, x, mask=None):
        self.x = x
        Q = x @ self.W_Q
        K = x @ self.W_K
        V = x @ self.W_V
        self.Q = self._split_heads(Q)
        self.K = self._split_heads(K)
        self.V = self._split_heads(V)
        scores = (self.Q @ self.K.transpose(0, 1, 3, 2)) / np.sqrt(self.d_k)
        if mask is not None:
            scores = np.where(mask, -1e9, scores)
        self.attn_weights = softmax(scores)
        attn_out = self.attn_weights @ self.V
        self.attn_output = self._merge_heads(attn_out)
        return self.attn_output @ self.W_O
    
    def backward(self, dout):
        B, T, D = dout.shape
        self.dW_O = self.attn_output.reshape(-1, D).T @ dout.reshape(-1, D)
        d_attn_output = dout @ self.W_O.T
        d_attn_out = self._split_heads(d_attn_output)
        d_attn_weights = d_attn_out @ self.V.transpose(0, 1, 3, 2)
        dV = self.attn_weights.transpose(0, 1, 3, 2) @ d_attn_out
        sum_term = np.sum(d_attn_weights * self.attn_weights, axis=-1, keepdims=True)
        d_scores = self.attn_weights * (d_attn_weights - sum_term)
        d_scores /= np.sqrt(self.d_k)
        dQ = d_scores @ self.K
        dK = d_scores.transpose(0, 1, 3, 2) @ self.Q
        dQ = self._merge_heads(dQ)
        dK = self._merge_heads(dK)
        dV = self._merge_heads(dV)
        x_flat = self.x.reshape(-1, D)
        self.dW_Q = x_flat.T @ dQ.reshape(-1, D)
        self.dW_K = x_flat.T @ dK.reshape(-1, D)
        self.dW_V = x_flat.T @ dV.reshape(-1, D)
        dx = dQ @ self.W_Q.T + dK @ self.W_K.T + dV @ self.W_V.T
        return dx


class FeedForward:
    def __init__(self, d_model, d_ff):
        scale1 = np.sqrt(2.0 / (d_model + d_ff))
        scale2 = np.sqrt(2.0 / (d_ff + d_model))
        self.W1 = np.random.randn(d_model, d_ff) * scale1
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * scale2
        self.b2 = np.zeros(d_model)
        self.dW1 = self.db1 = self.dW2 = self.db2 = None
        self.x = self.hidden_relu = self.relu_mask = None
    
    def forward(self, x):
        self.x = x
        hidden = x @ self.W1 + self.b1
        self.relu_mask = (hidden > 0).astype(float)
        self.hidden_relu = hidden * self.relu_mask
        return self.hidden_relu @ self.W2 + self.b2
    
    def backward(self, dout):
        d_model = dout.shape[-1]
        d_ff = self.W1.shape[1]
        self.dW2 = self.hidden_relu.reshape(-1, d_ff).T @ dout.reshape(-1, d_model)
        self.db2 = dout.reshape(-1, d_model).sum(axis=0)
        d_hidden = (dout @ self.W2.T) * self.relu_mask
        self.dW1 = self.x.reshape(-1, d_model).T @ d_hidden.reshape(-1, d_ff)
        self.db1 = d_hidden.reshape(-1, d_ff).sum(axis=0)
        return d_hidden @ self.W1.T


class Linear:
    def __init__(self, in_features, out_features):
        scale = np.sqrt(2.0 / (in_features + out_features))
        self.W = np.random.randn(in_features, out_features) * scale
        self.b = np.zeros(out_features)
        self.dW = self.db = None
        self.x = None
    
    def forward(self, x):
        self.x = x
        return x @ self.W + self.b
    
    def backward(self, dout):
        self.dW = self.x.reshape(-1, self.x.shape[-1]).T @ dout.reshape(-1, dout.shape[-1])
        self.db = dout.reshape(-1, dout.shape[-1]).sum(axis=0)
        return dout @ self.W.T

print("All components defined.")

All components defined.


## 2. Transformer Block

In [4]:
class TransformerBlock:
    """A single transformer block: Pre-LN MHA + Pre-LN FFN with residuals."""
    
    def __init__(self, d_model, n_heads, d_ff):
        self.ln1 = LayerNorm(d_model)
        self.mha = MultiHeadAttention(d_model, n_heads)
        self.ln2 = LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)
        
        # Cache
        self.x = None
        self.x_after_attn = None
    
    def forward(self, x, mask=None):
        self.x = x
        
        # Sub-layer 1: LayerNorm -> MHA -> Residual
        x_norm = self.ln1.forward(x)
        attn_out = self.mha.forward(x_norm, mask=mask)
        self.x_after_attn = x + attn_out  # residual
        
        # Sub-layer 2: LayerNorm -> FFN -> Residual
        x_norm2 = self.ln2.forward(self.x_after_attn)
        ffn_out = self.ffn.forward(x_norm2)
        output = self.x_after_attn + ffn_out  # residual
        
        return output
    
    def backward(self, dout):
        # Backward through sub-layer 2
        d_ffn_out = dout  # residual path
        d_x_norm2 = self.ffn.backward(d_ffn_out)
        d_x_after_attn = dout + self.ln2.backward(d_x_norm2)  # residual + sublayer
        
        # Backward through sub-layer 1
        d_attn_out = d_x_after_attn  # residual path
        d_x_norm = self.mha.backward(d_attn_out)
        dx = d_x_after_attn + self.ln1.backward(d_x_norm)  # residual + sublayer
        
        return dx
    
    def get_params(self):
        """Return list of (param, grad) tuples for optimizer."""
        return [
            (self.ln1, 'gamma'), (self.ln1, 'beta'),
            (self.mha, 'W_Q'), (self.mha, 'W_K'),
            (self.mha, 'W_V'), (self.mha, 'W_O'),
            (self.ln2, 'gamma'), (self.ln2, 'beta'),
            (self.ffn, 'W1'), (self.ffn, 'b1'),
            (self.ffn, 'W2'), (self.ffn, 'b2'),
        ]

# Test
d_model, n_heads, d_ff = 64, 4, 256
block = TransformerBlock(d_model, n_heads, d_ff)
x = np.random.randn(2, 8, d_model)
mask = causal_mask(8)

out = block.forward(x, mask=mask)
print(f"TransformerBlock: {x.shape} -> {out.shape}")

dout = np.random.randn(2, 8, d_model)
dx = block.backward(dout)
print(f"Backward: dx shape = {dx.shape}")

TransformerBlock: (2, 8, 64) -> (2, 8, 64)
Backward: dx shape = (2, 8, 64)


## 3. Complete Decoder-Only Transformer

In [5]:
class Transformer:
    """Decoder-only transformer for character-level language modeling."""
    
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_seq_len):
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        
        # Token embedding
        self.embedding = Embedding(vocab_size, d_model)
        
        # Positional encoding (fixed, not learned)
        self.pe = get_positional_encoding(max_seq_len, d_model)
        
        # Transformer blocks
        self.blocks = [TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        
        # Final layer norm
        self.ln_final = LayerNorm(d_model)
        
        # Output projection to vocab
        self.output_proj = Linear(d_model, vocab_size)
    
    def forward(self, indices):
        """Forward pass.
        
        Args:
            indices: token indices, shape (B, T)
        Returns:
            logits: shape (B, T, vocab_size)
        """
        B, T = indices.shape
        
        # Embed tokens and add positional encoding
        x = self.embedding.forward(indices)  # (B, T, d_model)
        x = x + self.pe[:T]  # broadcast over batch
        
        # Create causal mask
        mask = causal_mask(T)
        
        # Pass through transformer blocks
        for block in self.blocks:
            x = block.forward(x, mask=mask)
        
        # Final layer norm
        x = self.ln_final.forward(x)
        
        # Project to vocabulary
        logits = self.output_proj.forward(x)  # (B, T, vocab_size)
        
        return logits
    
    def backward(self, dlogits):
        """Backward pass through entire model.
        
        Args:
            dlogits: gradient of shape (B, T, vocab_size)
        """
        # Backward through output projection
        dx = self.output_proj.backward(dlogits)
        
        # Backward through final layer norm
        dx = self.ln_final.backward(dx)
        
        # Backward through transformer blocks (reverse order)
        for block in reversed(self.blocks):
            dx = block.backward(dx)
        
        # Backward through embedding
        self.embedding.backward(dx)
    
    def get_params(self):
        """Get all (object, param_name) pairs for optimizer."""
        params = [(self.embedding, 'W')]
        for block in self.blocks:
            params.extend(block.get_params())
        params.extend([
            (self.ln_final, 'gamma'), (self.ln_final, 'beta'),
            (self.output_proj, 'W'), (self.output_proj, 'b'),
        ])
        return params
    
    def count_params(self):
        """Count total trainable parameters."""
        total = 0
        for obj, name in self.get_params():
            total += getattr(obj, name).size
        return total

print("Transformer model defined.")

Transformer model defined.


## 4. Instantiate and Inspect the Model

In [6]:
# Load data to get vocab size
with open('../data/input.txt', 'r') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

# Model hyperparameters
d_model = 64
n_heads = 4
d_ff = 256
n_layers = 3
max_seq_len = 32

model = Transformer(vocab_size, d_model, n_heads, d_ff, n_layers, max_seq_len)
print(f"Vocabulary size:    {vocab_size}")
print(f"Model dimension:    {d_model}")
print(f"Attention heads:    {n_heads}")
print(f"FFN dimension:      {d_ff}")
print(f"Transformer layers: {n_layers}")
print(f"Max sequence length: {max_seq_len}")
print(f"\nTotal parameters:   {model.count_params():,}")

Vocabulary size:    58
Model dimension:    64
Attention heads:    4
FFN dimension:      256
Transformer layers: 3
Max sequence length: 32

Total parameters:   156,794


In [7]:
# Test forward pass
def encode(s):
    return [char_to_idx[c] for c in s]

def decode(indices):
    return ''.join([idx_to_char[i] for i in indices])

data = np.array(encode(text), dtype=np.int64)

# Create a test batch
batch_size = 4
seq_len = 32
starts = np.random.randint(0, len(data) - seq_len - 1, size=batch_size)
x_batch = np.array([data[s:s+seq_len] for s in starts])
y_batch = np.array([data[s+1:s+seq_len+1] for s in starts])

# Forward pass
logits = model.forward(x_batch)
print(f"Input shape:  {x_batch.shape}")
print(f"Logits shape: {logits.shape}")

# Compute loss
logits_flat = logits.reshape(-1, vocab_size)
targets_flat = y_batch.reshape(-1)
loss, dlogits_flat = cross_entropy_loss(logits_flat, targets_flat)
print(f"\nInitial loss: {loss:.4f}")
print(f"Random guess loss (ln(vocab_size)): {np.log(vocab_size):.4f}")
print(f"Loss is close to random guess: model hasn't learned yet.")

Input shape:  (4, 32)
Logits shape: (4, 32, 58)

Initial loss: 4.9103
Random guess loss (ln(vocab_size)): 4.0604
Loss is close to random guess: model hasn't learned yet.


In [8]:
# Test backward pass
dlogits = dlogits_flat.reshape(batch_size, seq_len, vocab_size)
model.backward(dlogits)

# Verify gradients exist
print("Gradient check (all params have gradients):")
for obj, name in model.get_params():
    grad_name = 'd' + name
    grad = getattr(obj, grad_name)
    param = getattr(obj, name)
    if grad is None:
        print(f"  {name}: NO GRADIENT!")
    else:
        print(f"  {name}: shape={param.shape}, grad_norm={np.linalg.norm(grad):.4f}")

Gradient check (all params have gradients):
  W: shape=(58, 64), grad_norm=0.2114
  gamma: shape=(64,), grad_norm=0.2490
  beta: shape=(64,), grad_norm=0.3018
  W_Q: shape=(64, 64), grad_norm=0.2671
  W_K: shape=(64, 64), grad_norm=0.2790
  W_V: shape=(64, 64), grad_norm=1.7138
  W_O: shape=(64, 64), grad_norm=1.7048
  gamma: shape=(64,), grad_norm=0.1018
  beta: shape=(64,), grad_norm=0.1104
  W1: shape=(64, 256), grad_norm=1.3342
  b1: shape=(256,), grad_norm=0.1804
  W2: shape=(256, 64), grad_norm=1.3669
  b2: shape=(64,), grad_norm=0.1933
  gamma: shape=(64,), grad_norm=0.1831
  beta: shape=(64,), grad_norm=0.2061
  W_Q: shape=(64, 64), grad_norm=0.1028
  W_K: shape=(64, 64), grad_norm=0.0951
  W_V: shape=(64, 64), grad_norm=1.1589
  W_O: shape=(64, 64), grad_norm=1.2375
  gamma: shape=(64,), grad_norm=0.0621
  beta: shape=(64,), grad_norm=0.0726
  W1: shape=(64, 256), grad_norm=0.8916
  b1: shape=(256,), grad_norm=0.1171
  W2: shape=(256, 64), grad_norm=0.9612
  b2: shape=(64,), g

## 5. Parameter Breakdown

In [9]:
# Detailed parameter count
print(f"{'Component':<30} {'Parameters':>10}")
print("-" * 42)

emb_params = model.embedding.W.size
print(f"{'Embedding':<30} {emb_params:>10,}")

for i, block in enumerate(model.blocks):
    block_total = 0
    for obj, name in block.get_params():
        block_total += getattr(obj, name).size
    print(f"{'  Block ' + str(i):<30} {block_total:>10,}")

ln_params = model.ln_final.gamma.size + model.ln_final.beta.size
print(f"{'Final LayerNorm':<30} {ln_params:>10,}")

out_params = model.output_proj.W.size + model.output_proj.b.size
print(f"{'Output Projection':<30} {out_params:>10,}")

print("-" * 42)
print(f"{'TOTAL':<30} {model.count_params():>10,}")

Component                      Parameters
------------------------------------------
Embedding                           3,712
  Block 0                          49,728
  Block 1                          49,728
  Block 2                          49,728
Final LayerNorm                       128
Output Projection                   3,770
------------------------------------------
TOTAL                             156,794


## Summary

We've assembled a complete **decoder-only transformer** with:
- Character embedding + sinusoidal positional encoding
- 3 transformer blocks (Pre-LN MHA + FFN with residuals)
- Final LayerNorm + output projection
- Full forward and backward passes
- ~150K trainable parameters

Next notebook: **Training and Generation** - we'll train this model and generate text.